# 🔬 Entrenamiento YOLO11s — Versión 5 (Small)
## Finca Alvizua — Clasificación de 13 Clases de Café

Réplica exacta de los hiperparámetros de la **Versión 4 Campeona** (YOLO11m) con arquitectura **Small (~9.4M params)** para mejor relación parámetros/datos.

| Parámetro | V4 Campeona | V5 (Este notebook) |
| :--- | :---: | :---: |
| **Arquitectura** | YOLO11m (~20M) | **YOLO11s (~9.4M)** |
| **Optimizador** | AdamW + Cosine | AdamW + Cosine |
| **LR** | 0.001 | 0.001 |
| **Patience** | 35 | 35 |
| **imgsz** | 640 | 640 |

> ⚠️ **IMPORTANTE**: Antes de ejecutar, asegúrate de activar la GPU en Colab:
> `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`

---
## 1. Instalación de Dependencias

In [ ]:
!pip install ultralytics -q
!pip install pyyaml -q

---
## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 3. Verificar GPU Disponible

In [ ]:
!nvidia-smi
print("\n" + "=" * 50)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

---
## 4. Copiar y Descomprimir Dataset

Ajusta la ruta según donde tengas tu `dataset_yolo.zip` en Google Drive.

In [ ]:
import os

# === AJUSTA ESTA RUTA SEGÚN TU DRIVE ===
RUTA_ZIP_EN_DRIVE = "/content/drive/MyDrive/MOPPI/dataset_yolo.zip"

# Copiar al almacenamiento local de Colab (más rápido para entrenamiento)
if os.path.exists(RUTA_ZIP_EN_DRIVE):
    !cp "{RUTA_ZIP_EN_DRIVE}" /content/
    print("✅ ZIP copiado a /content/")
else:
    print(f"❌ No se encontró: {RUTA_ZIP_EN_DRIVE}")
    print("   Ajusta la variable RUTA_ZIP_EN_DRIVE")

# Descomprimir
!unzip -q -o /content/dataset_yolo.zip -d /content/
print("✅ Dataset descomprimido en /content/dataset_yolo/")

---
## 5. Verificar Estructura del Dataset

In [ ]:
import os

ruta_dataset = "/content/dataset_yolo"

# Verificar carpetas
for split in ["train", "val"]:
    imgs = os.path.join(ruta_dataset, "images", split)
    lbls = os.path.join(ruta_dataset, "labels", split)
    n_imgs = len(os.listdir(imgs)) if os.path.exists(imgs) else 0
    n_lbls = len(os.listdir(lbls)) if os.path.exists(lbls) else 0
    print(f"  {split}: {n_imgs} imágenes, {n_lbls} etiquetas")

print("\n📄 Contenido actual de data.yaml:")
!cat /content/dataset_yolo/data.yaml

---
## 6. Actualizar Rutas de data.yaml para Colab

In [ ]:
import yaml

data_yaml_path = "/content/dataset_yolo/data.yaml"

with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

# Actualizar path raíz para Colab
data_config["path"] = "/content/dataset_yolo"
data_config["train"] = "images/train"
data_config["val"] = "images/val"

with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ data.yaml actualizado para Colab:")
print("─" * 40)
print(yaml.dump(data_config, default_flow_style=False))

---
## 7. 🚀 Entrenamiento YOLO11s

Hiperparámetros idénticos a la **Versión 4 Campeona**, solo cambia la arquitectura de Medium a **Small**.

In [ ]:
from ultralytics import YOLO

# Cargar modelo preentrenado YOLO11s (Small ~9.4M params)
modelo = YOLO("yolo11s.pt")

# Entrenar con hiperparámetros de la Versión 4 Campeona
resultados = modelo.train(
    # === Dataset ===
    data=data_yaml_path,
    imgsz=640,

    # === Duración y Early Stopping ===
    epochs=150,          # Techo alto; early stopping cortará antes
    patience=35,         # Misma paciencia que V4 campeona

    # === Optimizador: AdamW + Cosine Annealing (clave del éxito V4) ===
    optimizer="AdamW",
    lr0=0.001,           # Learning rate inicial
    lrf=0.01,            # Factor final del LR
    cos_lr=True,         # Decaimiento cosenoidal suave

    # === Regularización ===
    weight_decay=0.0005,
    dropout=0.0,

    # === Pesos de pérdida ===
    box=7.5,             # box_loss
    cls=0.5,             # cls_loss
    dfl=1.5,             # dfl_loss

    # === Aumento de datos ===
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.0,

    # === Hardware (Colab GPU) ===
    batch=-1,            # Auto-batch según VRAM de Colab
    workers=2,           # Colab funciona mejor con 2 workers
    device=0,            # GPU de Colab

    # === Proyecto y logging ===
    project="/content/runs",
    name="yolo11s_cafe_v5",
    exist_ok=True,
    verbose=True,
    save=True,
    save_period=25,      # Checkpoint cada 25 épocas
    plots=True,          # Generar gráficas de métricas

    # === Reproducibilidad ===
    seed=42,
)

---
## 8. 📊 Visualizar Resultados del Entrenamiento

In [ ]:
from IPython.display import Image, display
import os

ruta = "/content/runs/yolo11s_cafe_v5"

print("📈 Curvas de Entrenamiento:")
if os.path.exists(f"{ruta}/results.png"):
    display(Image(filename=f"{ruta}/results.png", width=900))
else:
    print("   No encontrado.")

In [ ]:
from IPython.display import Image, display

ruta = "/content/runs/yolo11s_cafe_v5"

print("🔢 Matriz de Confusión Normalizada:")
if os.path.exists(f"{ruta}/confusion_matrix_normalized.png"):
    display(Image(filename=f"{ruta}/confusion_matrix_normalized.png", width=700))
else:
    print("   No encontrado.")

In [ ]:
from IPython.display import Image, display

ruta = "/content/runs/yolo11s_cafe_v5"

print("🖼️ Predicciones sobre Validación:")
for nombre in ["val_batch0_pred.png", "val_batch1_pred.png", "val_batch2_pred.png"]:
    ruta_img = f"{ruta}/{nombre}"
    if os.path.exists(ruta_img):
        print(f"\n  {nombre}:")
        display(Image(filename=ruta_img, width=700))

---
## 9. ✅ Validación Final del Modelo Entrenado

In [ ]:
from ultralytics import YOLO

modelo_entrenado = YOLO("/content/runs/yolo11s_cafe_v5/weights/best.pt")
metricas = modelo_entrenado.val(data=data_yaml_path, imgsz=640, conf=0.30)

print("\n" + "=" * 50)
print("   MÉTRICAS FINALES YOLO11s — V5")
print("=" * 50)
print(f"   mAP50:     {metricas.box.map50:.4f} ({metricas.box.map50*100:.1f}%)")
print(f"   mAP50-95:  {metricas.box.map:.4f} ({metricas.box.map*100:.1f}%)")
print(f"   Precisión: {metricas.box.mp:.4f} ({metricas.box.mp*100:.1f}%)")
print(f"   Recall:    {metricas.box.mr:.4f} ({metricas.box.mr*100:.1f}%)")
print("=" * 50)

# Comparación con V4 Campeona (YOLO11m)
print("\n📊 Comparación con V4 Campeona (YOLO11m):")
print(f"   {'Métrica':<12} {'V4 (Medium)':>12} {'V5 (Small)':>12} {'Δ':>8}")
print(f"   {'─'*12} {'─'*12} {'─'*12} {'─'*8}")

v4 = {"map50": 0.5187, "map": 0.4453, "p": 0.494, "r": 0.583}
v5_map50 = metricas.box.map50
v5_map = metricas.box.map
v5_p = metricas.box.mp
v5_r = metricas.box.mr

print(f"   {'mAP50':<12} {v4['map50']*100:>11.1f}% {v5_map50*100:>11.1f}% {(v5_map50-v4['map50'])*100:>+7.1f}%")
print(f"   {'mAP50-95':<12} {v4['map']*100:>11.1f}% {v5_map*100:>11.1f}% {(v5_map-v4['map'])*100:>+7.1f}%")
print(f"   {'Precisión':<12} {v4['p']*100:>11.1f}% {v5_p*100:>11.1f}% {(v5_p-v4['p'])*100:>+7.1f}%")
print(f"   {'Recall':<12} {v4['r']*100:>11.1f}% {v5_r*100:>11.1f}% {(v5_r-v4['r'])*100:>+7.1f}%")

---
## 10. 💾 Guardar Modelo en Google Drive

In [ ]:
import shutil
import os

# Crear directorio en Drive
destino_drive = "/content/drive/MyDrive/MOPPI/modelos_entrenados"
os.makedirs(destino_drive, exist_ok=True)

# Copiar best.pt
origen_best = "/content/runs/yolo11s_cafe_v5/weights/best.pt"
destino_best = os.path.join(destino_drive, "yolo11s_best.pt")

if os.path.exists(origen_best):
    shutil.copy2(origen_best, destino_best)
    tamano_mb = os.path.getsize(destino_best) / (1024 * 1024)
    print(f"✅ Modelo guardado en Google Drive:")
    print(f"   {destino_best}")
    print(f"   Tamaño: {tamano_mb:.1f} MB")
else:
    print("❌ No se encontró best.pt. Verifica que el entrenamiento haya finalizado.")

# Copiar también last.pt como respaldo
origen_last = "/content/runs/yolo11s_cafe_v5/weights/last.pt"
destino_last = os.path.join(destino_drive, "yolo11s_last.pt")

if os.path.exists(origen_last):
    shutil.copy2(origen_last, destino_last)
    print(f"✅ Respaldo last.pt también guardado.")